### UseCase 1

for this example were the data is directly stored in s3 what are the best way to optimize in the below scenario
Your company collects millions of web server logs per day. Each log contains a timestamp, user ID, URL accessed, response time, and HTTP status code. Management wants to identify users who frequently encounter slow responses (>2s) or errors (5xx status codes), and generate a daily alert. Task: Explain how you would design a Spark job to process this data efficiently. Show pseudo code to calculate: Number of slow requests per user. Number of error requests per user. Explain how you would optimize this job for very large datasets (e.g., 100 million logs/day).


In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("WebLogAlertJob").getOrCreate()

# Read logs from S3 (Parquet recommended)
logs_df = spark.read.parquet("s3://web-logs/2025-09-24/") \
    .select("timestamp", "user_ID", "response_time", "HTTP_status_code")

# Filter slow responses and errors
filtered_df = logs_df.filter(
    (F.col("response_time") > 2) | (F.col("HTTP_status_code").between(500, 599))
)

# Aggregate per user
agg_df = filtered_df.groupBy("user_ID").agg(
    F.count("*").alias("total_flagged_requests"),
    F.sum(F.when(F.col("response_time") > 2, 1).otherwise(0)).alias("slow_requests"),
    F.sum(F.when(F.col("HTTP_status_code").between(500, 599), 1).otherwise(0)).alias("error_requests")
)

# Thresholding (optional)
alert_df = agg_df.filter((F.col("slow_requests") > 10) | (F.col("error_requests") > 5))

# Write alerts to S3
alert_df.write.mode("overwrite").parquet("s3://alerts/2025-09-24/")

In [ ]:
## Adding partitioning to the data for optmization

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

spark.conf.set("spark.sql.shuffle.partitions", "400")  # Based on cluster size


alert_df.write.partitionBy("date").parquet("s3://alerts/")

In [ ]:
## bucketting too 

df.write \
  .partitionBy("date") \
  .bucketBy(32, "user_ID") \
  .sortBy("user_ID") \
  .mode("overwrite") \
  .saveAsTable("bucketed_partitioned_logs")


## Note :  Bucketing only works when writing to a Hive-compatible table via .saveAsTable(). Glue datacatalog
## It won’t work with plain .parquet() writes to S3.

Since you're storing logs in S3 and processing daily alerts:
- ✅ Use partitioning by date for raw S3 storage
- ✅ Use bucketing by user_ID only if you're writing to a managed Hive table or Delta Lake


### Usecase 2

We have a Spark job that processes ~500M rows daily. This week, it ran 6× slower. No code changes. What went wrong?

I asked 3 questions:

 1️⃣ Did the data volume spike? → Nope. Same size. <br>
 2️⃣ Were there skewed partitions? → Yes. One partition had 80% of the data.<br>
 3️⃣ Was caching used correctly? → Nope. They cached a huge DataFrame… but never reused it.<br>
 
🔍 The Root Cause

<b>Data Skew:</b> One key had millions of rows, causing a single task to drag the job.<br>
Misused Cache: Memory was filled with unused cached data, forcing frequent GC pauses.

💡 The Fix

Applied salting to the skewed key before join → balanced partitions.<br>
Removed unnecessary cache() → freed executor memory.<br>
Added spark.sql.shuffle.partitions tuning → reduced shuffle overhead.<br>

⏱ Result

⏩ Job runtime: 3h → 28m<BR>
 💰 Cluster cost: -65%<BR>
 📈 SLA: Back on track.<BR>
⚡ Takeaway<BR>

In Spark, small mistakes scale into big problems.<br>
 Always check data distribution and cache usage before throwing more hardware at it.<br>
💬 Your Turn:<br>
 What’s the biggest Spark optimization win you’ve had? Drop it below — let’s build a playbook together.<br>

### Usecase 2

We have a Spark job that processes ~500M rows daily. This week, it ran 6× slower. No code changes. What went wrong?

I asked 3 questions:

 1️⃣ Did the data volume spike? → Nope. Same size. <br>
 2️⃣ Were there skewed partitions? → Yes. One partition had 80% of the data.<br>
 3️⃣ Was caching used correctly? → Nope. They cached a huge DataFrame… but never reused it.<br>
 
🔍 The Root Cause

<b>Data Skew:</b> One key had millions of rows, causing a single task to drag the job.<br>
Misused Cache: Memory was filled with unused cached data, forcing frequent GC pauses.

💡 The Fix

Applied salting to the skewed key before join → balanced partitions.<br>
Removed unnecessary cache() → freed executor memory.<br>
Added spark.sql.shuffle.partitions tuning → reduced shuffle overhead.<br>

⏱ Result

⏩ Job runtime: 3h → 28m<BR>
 💰 Cluster cost: -65%<BR>
 📈 SLA: Back on track.<BR>
⚡ Takeaway<BR>

In Spark, small mistakes scale into big problems.<br>
 Always check data distribution and cache usage before throwing more hardware at it.<br>
💬 Your Turn:<br>
 What’s the biggest Spark optimization win you’ve had? Drop it below — let’s build a playbook together.<br>